# Idea 1: route phases, separation over expected, Combine link
Route running for WR, built the same way as idea 2. Decisions and pre-specified tests are in `idea1/decisions.md`.

Needs 01_preprocess run first only for the Combine table; game tracking is processed here from the raw parquets (same fixes as 00/01). The game step takes about 20 minutes and saves per-season windows, so later runs can skip it.

In [ ]:
import sys
sys.path.insert(0, '..')

import gc
import warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.metrics import mean_squared_error
from shared.config import PROCESSED
from shared.io import load
from shared import preprocess as pp
from shared.drill_viewer import align_reps
from shared.reliability import split_half, spearman_brown
from shared.routes import launch, break_features, BREAK_ROUTES

warnings.filterwarnings('ignore')
CLEAN = PROCESSED / 'clean'
OUT = PROCESSED / 'idea1'
OUT.mkdir(exist_ok=True)
KEYS = ['game_id', 'play_id', 'nfl_id']
GF = ['launch_accel', 'speed_1s', 'entry_speed', 'break_retention', 'reaccel']

## Sample
WR by `nfl_position`, plays with a route, regular season, not nullified.

In [ ]:
plays = pp.na_to_null(load('player_play'))
games, players = load('games'), load('players')
d = (plays.merge(games[['game_id', 'season', 'season_type']], on='game_id')
          .merge(players[['nfl_id', 'nfl_position']], on='nfl_id'))
wr = d[d['nfl_position'].eq('WR') & d['route_ran'].notna()
       & d['season_type'].eq('REG') & d['play_nullified_by_penalty'].ne('Y')]
len(wr), wr['nfl_id'].nunique(), wr['route_ran'].value_counts().to_dict()

## Game windows (slow, run once)
Snap to 5 s, smoothed speed, acceleration and heading, glitch tracks dropped, direction flipped. Saves one file per season.

In [ ]:
RUN_WINDOWS = not (OUT / 'wr_win_2025.parquet').exists()

if RUN_WINDOWS:
    for yr in (2023, 2024, 2025):
        g = load(f'game_tracking_{yr}', columns=KEYS + ['time', 'x', 'y', 's', 'event']).merge(wr[KEYS], on=KEYS)
        g = pp.add_kinematics(g, KEYS)
        g = pp.flag_glitches(g, KEYS)
        g = g[~g['glitch_track']]
        g = pp.standardize_direction(g, plays)
        g = pp.smooth_kinematics(g, KEYS)
        g = pp.smooth_xy(g, KEYS)
        ev = (g[g['event'].isin(['ball_snap', 'pass_forward'])]
              .groupby(['game_id', 'play_id', 'event'])['time'].min().unstack())
        g = g.merge(ev.rename(columns={'ball_snap': 'st', 'pass_forward': 'pf'}),
                    left_on=['game_id', 'play_id'], right_index=True)
        g['t'] = (pd.to_datetime(g['time']) - pd.to_datetime(g['st'])).dt.total_seconds().round(1)
        g['t_pf'] = (pd.to_datetime(g['pf']) - pd.to_datetime(g['st'])).dt.total_seconds().round(1)
        g['pa'] = g.groupby(['game_id', 'play_id'])['event'].transform(
            lambda e: e.isin(['play_action', 'autoevent_playaction']).any())
        g = g[g['t'].between(0, 5)][KEYS + ['t', 't_pf', 'pa', 's_sg', 'a_sg', 'heading']]
        g.to_parquet(OUT / f'wr_win_{yr}.parquet', index=False)
        print(yr, len(g))
        del g
        gc.collect()

## Route phase features per play

In [ ]:
route = wr.set_index(KEYS)['route_ran']
rows = []
for yr in (2023, 2024, 2025):
    g = pd.read_parquet(OUT / f'wr_win_{yr}.parquet')
    for key, r in g.groupby(KEYS, sort=False):
        r = r.reset_index(drop=True)
        t_pf = r['t_pf'].iloc[0]
        if len(r) < 11 or r['t'].iloc[0] > 0.05:
            continue
        f = dict(zip(KEYS, key), t_pf=t_pf, play_action=bool(r['pa'].iloc[0]))
        f.update(launch(r))
        if route.get(key) in BREAK_ROUTES and pd.notna(t_pf) and t_pf >= 1.5:
            f.update(break_features(r[r['t'] <= t_pf], t_to=t_pf))
        rows.append(f)

feat = pd.DataFrame(rows).merge(
    wr[KEYS + ['season', 'route_ran', 'separation_at_pass_forward', 'cushion',
               'team_coverage_man_zone', 'lined_up_position', 'target']], on=KEYS)
feat.to_parquet(OUT / 'wr_feat.parquet', index=False)
feat[['t_pf'] + GF + ['break_t', 'turn_deg']].describe().round(2).T

In [ ]:
feat.groupby('route_ran')[['turn_deg', 'break_t', 'break_retention']].median().round(2)

## Separation over expected

In [ ]:
m = feat.dropna(subset=['separation_at_pass_forward', 'cushion', 'team_coverage_man_zone', 't_pf']).copy()
m['sep'] = m['separation_at_pass_forward']
m['slot'] = m['lined_up_position'].eq('SLOT_WR').astype(int)
m['man'] = m['team_coverage_man_zone'].eq('MAN_COVERAGE').astype(int)
m['pa'] = m['play_action'].astype(int)
n = m.groupby('nfl_id').size()
m = m[m['nfl_id'].isin(n[n >= 100].index)].copy()

EXP = 'sep ~ C(route_ran) + cushion + man + slot + t_pf + pa'
fit = smf.ols(EXP, m).fit()
m['soe'] = fit.resid
for c in GF:   # z-score within route type so routes are comparable
    m[c] = (m[c] - m.groupby('route_ran')[c].transform('mean')) / m.groupby('route_ran')[c].transform('std')
len(m), m['nfl_id'].nunique(), round(fit.rsquared, 3)

In [ ]:
pd.concat([split_half(m, 'nfl_id', ['soe']), split_half(m, 'nfl_id', GF)]).round(2)

## Test A: do route features predict separation for unseen receivers?

In [ ]:
gaps = []
for seed in range(20):
    rng = np.random.default_rng(seed)
    half_a = rng.random(len(m)) < 0.5
    style = m[half_a].groupby('nfl_id')[GF].mean()
    style = (style - style.mean()) / style.std()
    b = m[~half_a].drop(columns=GF).merge(style, left_on='nfl_id', right_index=True).reset_index(drop=True)
    ids = b['nfl_id'].unique(); rng.shuffle(ids)
    p_base, p_full = np.zeros(len(b)), np.zeros(len(b))
    for fold in np.array_split(ids, 5):
        test = b['nfl_id'].isin(fold).to_numpy()
        p_base[test] = smf.ols(EXP, b[~test]).fit().predict(b[test])
        p_full[test] = smf.ols(EXP + ' + ' + ' + '.join(GF), b[~test]).fit().predict(b[test])
    gaps.append(mean_squared_error(b['sep'], p_full) - mean_squared_error(b['sep'], p_base))
print('CV MSE, full minus base, mean', round(np.mean(gaps), 4), '| splits improved', sum(g < 0 for g in gaps), 'of 20')

## Combine route drills
Same features from each single-rep route drill, clip start as time 0, capped at 5 s. Z-scored within drill, averaged per player.

In [ ]:
cr = pd.read_parquet(PROCESSED / 'parquet' / 'nfl-big-data-bowl-2027__combine_results.parquet')
ct = pd.read_parquet(CLEAN / 'combine_tracking.parquet')
reps = align_reps(ct, cr.loc[cr['combine_position'].eq('WR'), 'nfl_id'])
reps['t'] = reps['t'].round(1)
reps = pp.smooth_xy(reps, ['event_id'])

BREAK_DRILLS = ['SLANT_ROUTE_LEFT', 'CURL_ROUTE_RIGHT', 'DAGGER_ROUTE_LEFT', 'SPEED_OUT_ROUTE_LEFT',
                'COMEBACK_ROUTE_RIGHT', 'SLOT_STRIKE_ROUTE_LEFT', 'SLOT_SAIL_ROUTE_RIGHT', 'POST_CORNER_ROUTE_RIGHT']
ROUTE_DRILLS = BREAK_DRILLS + ['GO_ROUTE_RIGHT']
rows = []
for e, c in reps[reps['drill_name'].isin(ROUTE_DRILLS)].groupby('event_id'):
    c = c.reset_index(drop=True)
    f = {'event_id': e, 'nfl_id': c['nfl_id'].iloc[0], 'drill': c['drill_name'].iloc[0]}
    f.update(launch(c))
    if f['drill'] in BREAK_DRILLS:
        f.update(break_features(c[c['t'] <= 5.0], t_to=5.0))
    rows.append(f)
comb = pd.DataFrame(rows)
comb.groupby('drill')[['launch_accel', 'entry_speed', 'break_retention', 'reaccel', 'break_t', 'turn_deg']].median().round(2)

In [ ]:
CF = ['launch_accel', 'entry_speed', 'break_retention', 'reaccel']
for f in CF:
    comb[f + '_z'] = (comb[f] - comb.groupby('drill')[f].transform('mean')) / comb.groupby('drill')[f].transform('std')

rel = {}
rng = np.random.default_rng(0)
for f in CF:
    wide = comb.pivot_table(index='nfl_id', columns='drill', values=f + '_z')
    rs = []
    for _ in range(200):
        cols = rng.permutation(wide.columns)
        a, b = wide[cols[: len(cols) // 2]].mean(axis=1), wide[cols[len(cols) // 2:]].mean(axis=1)
        ok = a.notna() & b.notna()
        rs.append(spearman_brown(np.corrcoef(a[ok], b[ok])[0, 1], 2))
    rel[f] = np.mean(rs)
pd.Series(rel).round(2)

## Test B: Combine tracking vs separation over expected
6 features, Holm correction.

In [ ]:
cf = comb.groupby('nfl_id')[[f + '_z' for f in CF]].mean()
cf['gauntlet_peak_speed'] = reps[reps['drill_name'].eq('GAUNTLET_DRILL')].groupby(['nfl_id', 'event_id'])['s_sg'].max().groupby('nfl_id').mean()
cf['forty_peak_speed'] = reps[reps['drill_name'].eq('FORTY_YARD_DASH')].groupby(['nfl_id', 'event_id'])['s_sg'].max().groupby('nfl_id').mean()

soe = m.groupby('nfl_id')['soe'].mean()
rows = []
for c in cf.columns:
    s = pd.concat([soe, cf[c]], axis=1, join='inner').dropna()
    r, p = stats.pearsonr(s[c], s['soe'])
    rows.append((c, len(s), r, p))
tb = pd.DataFrame(rows, columns=['feature', 'n', 'r', 'p']).sort_values('p').reset_index(drop=True)
k = len(tb)
tb['holm_p'] = np.minimum(1, np.maximum.accumulate([(k - i) * p for i, p in enumerate(tb['p'])]))
tb.round(3)